# RAGForge — Intelligent Engineering Knowledge Assistant

## Project Overview

RAGForge is an end-to-end Retrieval Augmented Generation (RAG) system
designed to answer questions using a domain-specific engineering
knowledge base.

Instead of relying only on an LLM's pre-trained knowledge, RAGForge
retrieves relevant information from external documents and provides
that information as context to the LLM before generating an answer.

The project progressively evolves from a simple keyword-based retrieval
system into an advanced RAG pipeline with semantic search, embeddings,
vector databases, query rewriting, reranking, intelligent chunking,
and automated evaluation.



In [ ]:
%pip install -q openai chromadb gradio pydantic scikit-learn matplotlib fpdf2 pypdf \
    langchain-core langchain-openai langchain-chroma langchain-text-splitters

In [ ]:
import os
import json
import math
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
import chromadb
import gradio as gr
import numpy as np
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# LangChain imports - used only in the "classic" section for comparison
from langchain_core.documents import Document
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

from fpdf import FPDF
from pypdf import PdfReader

In [ ]:
load_dotenv(override=True)

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
groq_client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=os.getenv("GROQ_API_KEY"))

MODEL = "gpt-4.1-mini"
GROQ_MODEL = "openai/gpt-oss-120b"
EMBEDDING_MODEL = "text-embedding-3-small"


def embed(texts):
    response = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in response.data]

## The knowledge base

Real PDF files, not text baked into Python. The notebook generates the PDFs itself into a
`knowledge-base/` folder (a relative path - no manual file setup needed), then reads them
back with real PDF text extraction, the same way the course's DirectoryLoader reads real
files from disk.

In [ ]:
KNOWLEDGE_BASE_DIR = "knowledge-base"
os.makedirs(KNOWLEDGE_BASE_DIR, exist_ok=True)

raw_documents = {
    "angular-components": "Every Angular component needs a TypeScript class for behavior, an HTML template, and a CSS selector defined via the @Component decorator. By default, components are standalone as of Angular 19+, meaning they can be added directly to another component's imports array without needing an NgModule. Angular renders one instance of a component for every matching HTML element, and templates can be written inline or split into separate .html/.css files using templateUrl and styleUrl.",

    "angular-di": "Dependency Injection lets a class receive the objects it depends on from the outside, rather than constructing them internally, which improves testability and reduces duplication. The modern pattern uses the @Service decorator (a shorthand for @Injectable({providedIn: 'root'})) to make a class injectable as an app-wide singleton, and the inject() function to retrieve dependencies - inject() can be called in a field initializer, a constructor body, or within a route guard, anywhere inside what Angular calls an 'injection context'.",

    "angular-signals": "Signals are Angular's fine-grained reactivity primitive, stabilized as of Angular 17 and now the recommended default for managing state. A signal wraps a value and notifies any code that reads it when that value changes, letting Angular update only the specific parts of the UI that actually depend on it rather than re-checking the whole component tree. Signals are synchronous by design; for state that depends on async data, Angular provides a separate Resource API instead.",

    "angular-change-detection": "Angular's default change detection strategy re-checks every component in the tree whenever any event fires anywhere in the app, which can become a performance bottleneck as an application grows. The OnPush strategy restricts a component to re-check only when an @Input reference changes, an event originates inside the component itself, or an observable bound via the async pipe emits - this requires treating inputs as immutable, since mutating an object in place won't be detected as a change.",

    "typescript-types": "TypeScript uses a structural type system: two types are considered compatible if they have the same shape, regardless of what they're named, unlike the nominal typing used in languages like Java. Interfaces and type aliases largely overlap for describing object shapes, but interfaces support declaration merging (multiple declarations combine automatically) while type aliases can additionally represent unions and intersections that interfaces cannot express.",

    "typescript-generics": "Generics let a function, class, or interface work across multiple types while preserving type information, instead of falling back to 'any' and losing type safety entirely. A generic function like function identity<T>(value: T): T infers T from whatever argument is passed in, so the compiler catches a type mismatch at compile time rather than the error only surfacing at runtime.",

    "rxjs-observables": "An Observable represents a stream of values over time and, unlike a Promise which resolves exactly once, can emit multiple values across its lifetime. Observables are cold by default - no code runs until something subscribes - and every subscription creates its own independent execution, so the same Observable subscribed to twice can trigger two separate underlying operations unless explicitly shared. Forgetting to unsubscribe from a long-lived Observable is one of the most common sources of memory leaks in Angular applications.",

    "rxjs-operators": "switchMap cancels the previous inner Observable the moment a new value arrives from the source, which is why it's the standard choice for search-as-you-type, where only the latest request's result matters. mergeMap instead runs every inner Observable concurrently with no cancellation, appropriate when every result is needed regardless of order. concatMap queues inner Observables and runs them strictly one at a time, preserving order at the cost of concurrency - useful for a sequence of writes that must happen in a specific order.",

    "adr-001-state-management": "ADR-001: State Management Approach. Status: Accepted. Context: as the application grew past 40 components, prop drilling and scattered subscriptions made state hard to trace. Decision: adopt a signal-based store pattern rather than a full external state library like NgRx, since the team's state needs are moderate and NgRx's boilerplate was judged not worth the added complexity. Consequences: a simpler mental model and less boilerplate, but less tooling support (no dev-tools time-travel debugging) than NgRx provides.",

    "onboarding-frontend": "Frontend Onboarding Guide. Week 1: clone the repo, run npm install, run ng serve, and get the local dev environment running against the staging API, plus a walkthrough of the shared component library and services folder. Week 2: pick up a ticket labeled 'good-first-issue' to get a full pull request through code review. Every PR requires at least one approval and a passing CI run (lint, unit tests, and a build check) before it can merge.",

    "troubleshooting-build-failures": "Troubleshooting: CI Build Failures. If a build fails with a type error that doesn't reproduce locally, first check that your local TypeScript version matches the version pinned in package.json - version drift is the most common cause of this exact symptom. If tests fail intermittently in CI but pass locally every time, check for tests that depend on execution order or share mutable state between test files, since CI often runs tests in a different order than local test runners default to.",
}


def create_pdf(filepath, title, text):
    """Write a simple, real PDF file - a title line plus the body text."""
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 14)
    pdf.multi_cell(0, 10, title)
    pdf.ln(4)
    pdf.set_font("Helvetica", size=11)
    pdf.multi_cell(0, 7, text)
    pdf.output(filepath)


for name, text in raw_documents.items():
    filepath = os.path.join(KNOWLEDGE_BASE_DIR, f"{name}.pdf")
    title = name.replace("-", " ").title()
    create_pdf(filepath, title, text)

print(f"Created {len(raw_documents)} PDF files in {KNOWLEDGE_BASE_DIR}/")

In [ ]:
def read_pdf(filepath):
    """Extract the text content of a real PDF file - genuine document ingestion,
    the same operation the course's DirectoryLoader does for you automatically."""
    reader = PdfReader(filepath)
    text = "\n".join(page.extract_text() for page in reader.pages)
    return text.strip()


knowledge_base = {}
for filename in sorted(os.listdir(KNOWLEDGE_BASE_DIR)):
    if filename.endswith(".pdf"):
        name = filename.replace(".pdf", "")
        filepath = os.path.join(KNOWLEDGE_BASE_DIR, filename)
        knowledge_base[name] = read_pdf(filepath)

for name, text in knowledge_base.items():
    print(f"--- {name} ({len(text)} chars, read from {name}.pdf) ---")
    print(text[:100] + "...")
    print()

## naive keyword search

In [ ]:
def naive_search(query):
    query_words = query.lower().split()
    matches = []
    for name, text in knowledge_base.items():
        if any(word in text.lower() for word in query_words):
            matches.append(name)
    return matches


print(naive_search("state management"))
print(naive_search("build broken"))

## The LangChain way 

LangChain handles document representation, chunking, embeddings, and retrieval through its own abstractions.

In [ ]:
langchain_documents = [
    Document(page_content=text, metadata={"source": name})
    for name, text in knowledge_base.items()
]

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
langchain_chunks = splitter.split_documents(langchain_documents)

print(f"LangChain produced {len(langchain_chunks)} chunks (fixed-size splitting, no LLM involved)")

langchain_embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
langchain_vectorstore = Chroma.from_documents(langchain_chunks, embedding=langchain_embeddings)
langchain_retriever = langchain_vectorstore.as_retriever(search_kwargs={"k": 3})

langchain_llm = ChatOpenAI(model=MODEL, temperature=0)

print("LangChain vector store ready")

In [ ]:
LANGCHAIN_SYSTEM_PROMPT = """You are an internal engineering knowledge assistant.
Use the given context to answer the question. If you don't know, say so.
Context:
{context}
"""


def answer_langchain(question):
    docs = langchain_retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = LANGCHAIN_SYSTEM_PROMPT.format(context=context)
    messages = [SystemMessage(content=system_prompt), HumanMessage(content=question)]
    response = langchain_llm.invoke(messages)
    sources = list(set(doc.metadata["source"] for doc in docs))
    return response.content, sources


answer, sources = answer_langchain("Why did the team choose signals over NgRx?")
print("Answer:", answer)
print("Sources:", sources)

## The native way 
Instead of a fixed character count, an LLM reads each document and decides where the real
conceptual breaks are - the more advanced approach.

In [ ]:
class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, a few words, likely to match a query")
    summary: str = Field(description="One sentence summarizing this chunk")
    original_text: str = Field(description="The original text, unchanged")


class Chunks(BaseModel):
    chunks: list[Chunk]


def llm_chunk_document(name, text):
    response = openai_client.chat.completions.parse(
        model=MODEL,
        messages=[{
            "role": "user",
            "content": f"Split this document into 1-3 chunks for a knowledge base search system. "
                       f"Each chunk should be a coherent, self-contained piece of information. "
                       f"Document:\n\n{text}"
        }],
        response_format=Chunks,
    )
    parsed = Chunks.model_validate_json(response.choices[0].message.content)
    return [(f"{c.headline}\n\n{c.summary}\n\n{c.original_text}", name) for c in parsed.chunks]


all_chunks = []
chunk_sources = []
for name, text in knowledge_base.items():
    for chunk_text, source in llm_chunk_document(name, text):
        all_chunks.append(chunk_text)
        chunk_sources.append(source)

print(f"Native LLM-based chunking produced {len(all_chunks)} chunks (vs {len(langchain_chunks)} from LangChain's fixed-size splitter)")

In [ ]:
chroma_client = chromadb.Client()
collection = chroma_client.create_collection("ragforge_docs")

chunk_embeddings = embed(all_chunks)
collection.add(
    ids=[str(i) for i in range(len(all_chunks))],
    documents=all_chunks,
    embeddings=chunk_embeddings,
    metadatas=[{"source": s} for s in chunk_sources],
)

print(f"{collection.count()} chunks stored in Chroma (native, in-memory)")

## visualizing the embeddings

t-SNE projects the high-dimensional embedding vectors down to 2D so you can actually see
whether chunks about similar topics cluster together in space.

In [ ]:
def plot_embeddings():
    vectors = np.array(chunk_embeddings)
    n_samples = len(vectors)
    perplexity = min(5, max(1, n_samples - 1))

    tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42)
    reduced = tsne.fit_transform(vectors)

    unique_sources = sorted(set(chunk_sources))
    colors = plt.cm.tab10(np.linspace(0, 1, len(unique_sources)))
    color_map = dict(zip(unique_sources, colors))

    fig, ax = plt.subplots(figsize=(8, 6))
    for source in unique_sources:
        idx = [i for i, s in enumerate(chunk_sources) if s == source]
        ax.scatter(reduced[idx, 0], reduced[idx, 1], label=source, color=color_map[source], s=80)

    ax.set_title("Chunk embeddings, projected to 2D")
    ax.legend(fontsize=7, loc="best")
    return fig


fig = plot_embeddings()
plt.show()

## native retrieval + a grounded answer

In [ ]:
SYSTEM_PROMPT = """You are an internal engineering knowledge assistant covering frontend
development (Angular, TypeScript, RxJS) and team engineering process (ADRs, onboarding,
troubleshooting). Answer using only the provided context. Be precise. If the context doesn't
cover the question, say so honestly. Keep answers to 2-4 sentences."""


def retrieve(question, k=3):
    query_embedding = embed([question])
    results = collection.query(query_embeddings=query_embedding, n_results=k)
    chunks = results["documents"][0]
    sources = [m["source"] for m in results["metadatas"][0]]
    return chunks, sources


def answer_basic(question, client=openai_client, model=MODEL, k=3):
    chunks, sources = retrieve(question, k=k)
    context = "\n\n".join(chunks)
    response = client.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ]
    )
    return response.choices[0].message.content, list(set(sources))

## query rewriting + reranking

In [ ]:
def rewrite_query(question, client=openai_client, model=MODEL):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "Rewrite this question as a short, specific search "
                                           "query likely to match relevant technical documentation. "
                                           "Reply with only the rewritten query."},
            {"role": "user", "content": question}
        ]
    )
    return response.choices[0].message.content.strip()


class RankOrder(BaseModel):
    order: list[int] = Field(description="Chunk ids ordered from most to least relevant")


def rerank(question, chunks, sources, client=openai_client, model=MODEL):
    numbered = "\n".join(f"CHUNK {i}:\n{c}" for i, c in enumerate(chunks))
    response = client.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": "Rank these chunks by relevance to the question, "
                                           "most relevant first. Include every chunk id."},
            {"role": "user", "content": f"Question: {question}\n\n{numbered}"}
        ],
        response_format=RankOrder,
    )
    order = RankOrder.model_validate_json(response.choices[0].message.content).order
    order = [i for i in order if 0 <= i < len(chunks)]
    return [chunks[i] for i in order], [sources[i] for i in order]


def merge_unique(chunks_a, sources_a, chunks_b, sources_b):
    seen = set(chunks_a)
    merged_chunks, merged_sources = list(chunks_a), list(sources_a)
    for c, s in zip(chunks_b, sources_b):
        if c not in seen:
            merged_chunks.append(c)
            merged_sources.append(s)
            seen.add(c)
    return merged_chunks, merged_sources


def answer_smart(question, client=openai_client, model=MODEL, k=4):
    rewritten = rewrite_query(question, client, model)
    chunks_a, sources_a = retrieve(question, k=k)
    chunks_b, sources_b = retrieve(rewritten, k=k)
    chunks, sources = merge_unique(chunks_a, sources_a, chunks_b, sources_b)
    chunks, sources = rerank(question, chunks, sources, client, model)
    chunks, sources = chunks[:3], sources[:3]

    context = "\n\n".join(chunks)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ]
    )
    return response.choices[0].message.content, list(set(sources))

## evaluation

In [ ]:
test_questions = [
    {"question": "Why did the team choose signals over NgRx for state management?",
     "keywords": ["signals", "NgRx", "boilerplate"]},
    {"question": "What's the first thing to check if CI build fails but works locally?",
     "keywords": ["TypeScript", "version", "package.json"]},
    {"question": "When should I use switchMap instead of mergeMap?",
     "keywords": ["switchMap", "cancel", "search"]},
    {"question": "How does Angular's OnPush strategy reduce unnecessary checks?",
     "keywords": ["OnPush", "Input", "async pipe"]},
    {"question": "What's required before a PR can merge?",
     "keywords": ["approval", "CI", "lint"]},
]


def calculate_mrr(keyword, chunks):
    for rank, chunk in enumerate(chunks, start=1):
        if keyword.lower() in chunk.lower():
            return 1.0 / rank
    return 0.0


def calculate_ndcg(keyword, chunks, k=10):
    relevances = [1 if keyword.lower() in c.lower() else 0 for c in chunks[:k]]
    dcg = sum(r / math.log2(i + 2) for i, r in enumerate(relevances))
    ideal = sorted(relevances, reverse=True)
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def evaluate_retrieval(test_questions, retrieve_fn):
    mrr_scores, ndcg_scores = [], []
    for test in test_questions:
        chunks, _ = retrieve_fn(test["question"])
        mrr_scores.append(sum(calculate_mrr(k, chunks) for k in test["keywords"]) / len(test["keywords"]))
        ndcg_scores.append(sum(calculate_ndcg(k, chunks) for k in test["keywords"]) / len(test["keywords"]))
    return sum(mrr_scores) / len(mrr_scores), sum(ndcg_scores) / len(ndcg_scores)


def smart_retrieve(question, k=4):
    rewritten = rewrite_query(question)
    ca, sa = retrieve(question, k=k)
    cb, sb = retrieve(rewritten, k=k)
    chunks, sources = merge_unique(ca, sa, cb, sb)
    chunks, sources = rerank(question, chunks, sources)
    return chunks[:3], sources[:3]


def langchain_retrieve(question, k=3):
    docs = langchain_retriever.invoke(question)
    return [d.page_content for d in docs], [d.metadata["source"] for d in docs]


basic_mrr, basic_ndcg = evaluate_retrieval(test_questions, retrieve)
smart_mrr, smart_ndcg = evaluate_retrieval(test_questions, smart_retrieve)
langchain_mrr, langchain_ndcg = evaluate_retrieval(test_questions, langchain_retrieve)

print(f"LangChain (classic) - MRR: {langchain_mrr:.3f}  nDCG: {langchain_ndcg:.3f}")
print(f"Native basic         - MRR: {basic_mrr:.3f}  nDCG: {basic_ndcg:.3f}")
print(f"Native smart         - MRR: {smart_mrr:.3f}  nDCG: {smart_ndcg:.3f}")

In [ ]:
class AnswerEval(BaseModel):
    feedback: str = Field(description="Concise feedback on the answer")
    accuracy: float = Field(description="1 (wrong) to 5 (perfectly accurate)")
    completeness: float = Field(description="1 (missing info) to 5 (fully complete)")
    relevance: float = Field(description="1 (off-topic) to 5 (directly on-topic)")


def judge_answer(question, generated_answer, reference_answer):
    response = openai_client.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are an expert evaluator. Only give 5/5 for a truly perfect answer."},
            {"role": "user", "content": f"Question: {question}\n\nGenerated: {generated_answer}\n\n"
                                        f"Reference: {reference_answer}\n\nScore accuracy, completeness, relevance (1-5)."}
        ],
        response_format=AnswerEval,
    )
    return AnswerEval.model_validate_json(response.choices[0].message.content)


question = "Why did the team choose signals over NgRx?"
reference = "The team chose signals because NgRx's boilerplate wasn't worth it for their moderate state needs."
generated, _ = answer_smart(question)
result = judge_answer(question, generated, reference)

print(f"Accuracy: {result.accuracy}/5, Completeness: {result.completeness}/5, Relevance: {result.relevance}/5")
print(f"Feedback: {result.feedback}")

In [ ]:
def ask(question, mode, provider):
    client, model = (openai_client, MODEL) if provider == "GPT" else (groq_client, GROQ_MODEL)

    if mode == "Naive keyword search":
        sources = naive_search(question)
        answer = "\n\n".join(knowledge_base[s][:250] for s in sources) if sources else "No matches found."
        return answer, ", ".join(sources)

    if mode == "LangChain (classic)":
        answer, sources = answer_langchain(question)
        return answer, ", ".join(sources)

    if mode == "Native basic RAG":
        answer, sources = answer_basic(question, client=client, model=model)
    else:
        answer, sources = answer_smart(question, client=client, model=model)

    return answer, ", ".join(sources)


with gr.Blocks(title="RAGForge") as ui:
    gr.Markdown("# RAGForge")
    gr.Markdown("Internal engineering knowledge assistant - compare LangChain vs. native RAG.")

    with gr.Row():
        provider = gr.Dropdown(["GPT", "Groq"], value="GPT", label="Model")
        mode = gr.Radio(
            ["Naive keyword search", "LangChain (classic)", "Native basic RAG", "Native smart RAG"],
            value="Native smart RAG",
            label="Retrieval mode"
        )

    question_input = gr.Textbox(label="Question", placeholder="Why did we choose signals over NgRx?")
    ask_button = gr.Button("Ask", variant="primary")

    answer_output = gr.Textbox(label="Answer", lines=4)
    sources_output = gr.Textbox(label="Sources", lines=1)

    ask_button.click(ask, inputs=[question_input, mode, provider], outputs=[answer_output, sources_output])

ui.launch(share=True)